
# Aula Prática - Comparação de Classificadores

**Curso:** Ciência da Computação - Uniderp

**Tema:** Aprendizado de Máquina + NLU/NLG  

**Ambiente:** Jupyter Notebook / JupyterLab / Google Colab

**Professor:** Murilo Gustavo Nabarrete Costa

## Objetivo
Comparar o desempenho de diferentes classificadores em uma tarefa de NLU (Natural Language Understanding).



### Intenções

Nosso chatbot terá quatro intenções:

- `saudacao`
- `preco`
- `estoque`
- `pedido`

Exemplos:

| Mensagem | Intenção |
|---|---|
| "Oi" | saudacao |
| "Quanto custa o notebook?" | preco |
| "Tem o celular em estoque?" | estoque |
| "Quero saber onde está meu pedido" | pedido |

A ideia central é que **mensagens diferentes podem representar a mesma intenção**.


In [ ]:
#Preparação do ambiente Local

# Se alguma biblioteca não estiver instalada, execute esta célula.
%pip install -q pandas scikit-learn matplotlib


In [ ]:

import re
import random
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report



## Criando os dados de treinamento

Em **aprendizado supervisionado**, o modelo recebe exemplos de entrada associados a uma saída conhecida.

Aqui:

- entrada = mensagem do usuário;
- saída = intenção correta.

Esse conjunto pequeno será nosso **dataset de treinamento**.

> Em um sistema real, precisaríamos de muito mais exemplos e de uma estratégia cuidadosa para construção e validação do dataset.


In [ ]:
dados = [
    # ============================================================
    # SAUDACAO - 30 exemplos
    # ============================================================
    ("oi", "saudacao"),
    ("olá", "saudacao"),
    ("bom dia", "saudacao"),
    ("boa tarde", "saudacao"),
    ("boa noite", "saudacao"),
    ("oi tudo bem", "saudacao"),
    ("olá gostaria de ajuda", "saudacao"),
    ("oi, tudo certo", "saudacao"),
    ("olá, tudo bem", "saudacao"),
    ("bom dia, preciso de ajuda", "saudacao"),
    ("boa tarde, poderia me ajudar", "saudacao"),
    ("boa noite, preciso de uma informação", "saudacao"),
    ("oi, posso tirar uma dúvida", "saudacao"),
    ("olá, posso fazer uma pergunta", "saudacao"),
    ("oi, preciso de ajuda", "saudacao"),
    ("olá, gostaria de falar com vocês", "saudacao"),
    ("bom dia, tudo bem", "saudacao"),
    ("boa tarde, tudo bem", "saudacao"),
    ("boa noite, tudo bem", "saudacao"),
    ("oi, gostaria de uma informação", "saudacao"),
    ("olá, preciso de uma informação", "saudacao"),
    ("oi, vocês podem me ajudar", "saudacao"),
    ("olá, podem me ajudar", "saudacao"),
    ("bom dia, podem me ajudar", "saudacao"),
    ("boa tarde, tenho uma dúvida", "saudacao"),
    ("boa noite, tenho uma pergunta", "saudacao"),
    ("oi, gostaria de ajuda com uma compra", "saudacao"),
    ("olá, quero tirar uma dúvida", "saudacao"),
    ("oi, posso pedir uma informação", "saudacao"),
    ("olá, gostaria de atendimento", "saudacao"),

    # ============================================================
    # PRECO - 30 exemplos
    # ============================================================
    ("qual o preço do notebook", "preco"),
    ("quanto custa o celular", "preco"),
    ("qual o valor desse produto", "preco"),
    ("quero saber o preço do computador", "preco"),
    ("me informe o valor do smartphone", "preco"),
    ("quanto vou pagar pelo notebook", "preco"),
    ("esse produto custa quanto", "preco"),
    ("quanto custa esse produto", "preco"),
    ("qual é o valor do notebook", "preco"),
    ("qual é o preço desse celular", "preco"),
    ("quanto custa o computador", "preco"),
    ("gostaria de saber o preço", "preco"),
    ("pode me informar o valor", "preco"),
    ("qual o valor do celular", "preco"),
    ("quanto custa esse notebook", "preco"),
    ("quero saber quanto custa", "preco"),
    ("poderia informar o preço", "preco"),
    ("me diga quanto custa o produto", "preco"),
    ("qual o preço desse computador", "preco"),
    ("quanto vou pagar por esse celular", "preco"),
    ("qual o valor desse notebook", "preco"),
    ("gostaria de saber quanto custa o smartphone", "preco"),
    ("esse notebook está quanto", "preco"),
    ("esse celular custa quanto", "preco"),
    ("quanto está custando o produto", "preco"),
    ("qual é o preço do smartphone", "preco"),
    ("quanto custa para comprar esse produto", "preco"),
    ("me passa o preço do notebook", "preco"),
    ("pode me dizer o valor do celular", "preco"),
    ("quero consultar o preço do produto", "preco"),

    # ============================================================
    # ESTOQUE - 30 exemplos
    # ============================================================
    ("tem notebook em estoque", "estoque"),
    ("o celular está disponível", "estoque"),
    ("vocês têm esse produto", "estoque"),
    ("tem o smartphone disponível", "estoque"),
    ("quero saber se o computador está em estoque", "estoque"),
    ("esse produto ainda está disponível", "estoque"),
    ("posso comprar esse produto agora", "estoque"),
    ("o notebook está disponível", "estoque"),
    ("vocês têm celular em estoque", "estoque"),
    ("tem esse computador disponível", "estoque"),
    ("ainda tem esse produto", "estoque"),
    ("esse celular está disponível para compra", "estoque"),
    ("tem smartphone em estoque", "estoque"),
    ("o produto está disponível", "estoque"),
    ("vocês ainda têm esse notebook", "estoque"),
    ("quero saber se tem celular disponível", "estoque"),
    ("esse computador ainda está em estoque", "estoque"),
    ("tem disponibilidade desse produto", "estoque"),
    ("o notebook ainda está disponível", "estoque"),
    ("vocês têm esse celular disponível", "estoque"),
    ("posso comprar esse notebook", "estoque"),
    ("esse produto está disponível para compra", "estoque"),
    ("ainda existe esse produto em estoque", "estoque"),
    ("tem esse modelo disponível", "estoque"),
    ("quero saber se o produto está disponível", "estoque"),
    ("há notebooks disponíveis", "estoque"),
    ("há celulares em estoque", "estoque"),
    ("esse smartphone está disponível", "estoque"),
    ("vocês possuem esse produto em estoque", "estoque"),
    ("é possível comprar esse produto agora", "estoque"),

    # ============================================================
    # PEDIDO - 30 exemplos
    # ============================================================
    ("onde está meu pedido", "pedido"),
    ("quero acompanhar meu pedido", "pedido"),
    ("qual o status da minha entrega", "pedido"),
    ("meu pedido já foi enviado", "pedido"),
    ("quando meu pedido vai chegar", "pedido"),
    ("quero rastrear minha compra", "pedido"),
    ("como acompanho a entrega", "pedido"),
    ("onde está a minha compra", "pedido"),
    ("como posso rastrear meu pedido", "pedido"),
    ("qual a situação do meu pedido", "pedido"),
    ("quero saber onde está meu pedido", "pedido"),
    ("quando minha compra será entregue", "pedido"),
    ("me informe o status do pedido", "pedido"),
    ("meu pedido foi enviado", "pedido"),
    ("como faço para acompanhar meu pedido", "pedido"),
    ("quero acompanhar a entrega", "pedido"),
    ("tem como rastrear minha compra", "pedido"),
    ("quando meu pedido chega", "pedido"),
    ("gostaria de rastrear meu pedido", "pedido"),
    ("quero saber quando vai chegar", "pedido"),
    ("onde posso acompanhar minha entrega", "pedido"),
    ("meu pedido está a caminho", "pedido"),
    ("como vejo o status da minha compra", "pedido"),
    ("quero consultar meu pedido", "pedido"),
    ("pode me informar onde está meu pedido", "pedido"),
    ("qual a previsão de entrega do meu pedido", "pedido"),
    ("quero saber a previsão de entrega", "pedido"),
    ("como faço para rastrear a entrega", "pedido"),
    ("me diga o status da minha compra", "pedido"),
    ("quero verificar minha entrega", "pedido"),
]

df = pd.DataFrame(dados, columns=["texto", "intencao"])

df


## Conhecendo os dados

Antes de treinar um modelo, observe os exemplos.

**Discussão:** essa abordagem baseada em regras pode funcionar para exemplos simples, mas tende a falhar quando a mesma intenção é expressa de várias maneiras.

É justamente aí que entra a classificação de intenções.


In [ ]:

print(df["intencao"].value_counts())



## Separando treinamento e teste

Vamos separar os exemplos em dois grupos:

- **treinamento:** exemplos utilizados pelo modelo para aprender;
- **teste:** exemplos reservados para verificar como o modelo se comporta diante de dados que não foram utilizados no treinamento.

Como o dataset é pequeno, esta divisão é apenas didática.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    df["texto"],
    df["intencao"],
    test_size=0.25,
    random_state=42,
    stratify=df["intencao"]
)

print("Treinamento:", len(X_train))
print("Teste:", len(X_test))



## Transformando texto em números

Computadores e algoritmos de ML não trabalham diretamente com o significado humano das frases. Precisamos transformar o texto em uma representação numérica.

Vamos utilizar:

**CountVectorizer → LogisticRegression**

O `CountVectorizer` transforma as palavras em características numéricas com base em sua ocorrência nos textos.


A `LogisticRegression` será utilizada como classificador das intenções.


In [ ]:

modelo = Pipeline([
    ("vetorizador", CountVectorizer()),
    ("classificador", LogisticRegression(max_iter=1000))
])

modelo.fit(X_train, y_train)

print("Modelo treinado!")


A `DecisionTreeClassifier` será utilizada como classificador das intenções.


In [ ]:
modelo_arvore = Pipeline([
    ("vetorizador", CountVectorizer()),
    ("classificador", DecisionTreeClassifier(
        max_depth=8,
        random_state=42
    ))
])

modelo_arvore.fit(X_train, y_train)

print("Árvore de decisão treinada!")


## Testando o classificador

Agora vamos fornecer mensagens que o modelo não recebeu exatamente dessa forma durante o treinamento.

Observe principalmente a **intenção prevista**.


In [ ]:
mensagens = [
    "quanto custa o smartphone",
    "vocês têm notebook disponível?",
    "quero acompanhar a minha entrega",
    "olá, preciso de ajuda",
    "qual o valor do computador?",
    "meu pedido está a caminho",
    "tem celular em estoque?",
    "boa tarde, gostaria de uma informação",
]

# Previsões dos dois modelos
previsoes_lr = modelo.predict(mensagens)
previsoes_arvore = modelo_arvore.predict(mensagens)

for mensagem, previsao_lr, previsao_arvore in zip(
    mensagens,
    previsoes_lr,
    previsoes_arvore
):

    print(f"Mensagem: {mensagem}")
    print(f"Regressão Logística : {previsao_lr}")
    print(f"Árvore de Decisão  : {previsao_arvore}")

    if previsao_lr == previsao_arvore:
        print("Resultado: ✓ Modelos concordam")
    else:
        print("Resultado: ⚠ Modelos discordam")

    print("-" * 60)

### Teste de processo completo detalhado

In [ ]:
mensagem = "quanto custa o notebook?"

X_lr = modelo.named_steps["vetorizador"].transform([mensagem])
prob_lr = modelo.named_steps["classificador"].predict_proba(X_lr)[0]

X_arvore = modelo_arvore.named_steps["vetorizador"].transform([mensagem])
prob_arvore = modelo_arvore.named_steps["classificador"].predict_proba(X_arvore)[0]

print("Mensagem:", mensagem)

print("\n=== REGRESSÃO LOGÍSTICA ===")

for classe, probabilidade in zip(
    modelo.named_steps["classificador"].classes_,
    prob_lr
):
    print(f"{classe:10} → {probabilidade:.2%}")

print(
    "Intenção:",
    modelo.named_steps["classificador"].classes_[prob_lr.argmax()]
)


print("\n=== ÁRVORE DE DECISÃO ===")

for classe, probabilidade in zip(
    modelo_arvore.named_steps["classificador"].classes_,
    prob_arvore
):
    print(f"{classe:10} → {probabilidade:.2%}")

print(
    "Intenção:",
    modelo_arvore.named_steps["classificador"].classes_[prob_arvore.argmax()]
)

### O modelo aprende pesos associados às características. Esses pesos contribuem para decidir qual classe é mais provável.

### Regressão Logística:

"Quais palavras têm maior peso para cada intenção?"

In [ ]:
classificador_lr = modelo.named_steps["classificador"]
vetorizador_lr = modelo.named_steps["vetorizador"]

features_lr = vetorizador_lr.get_feature_names_out()

pesos_df = pd.DataFrame(
    classificador_lr.coef_,
    columns=features_lr,
    index=classificador_lr.classes_
)

pesos_df

### Árvore de Decisão:

"Quais palavras foram mais importantes nas decisões da árvore?"

In [ ]:
classificador_arvore = modelo_arvore.named_steps["classificador"]
vetorizador_arvore = modelo_arvore.named_steps["vetorizador"]

features_arvore = vetorizador_arvore.get_feature_names_out()

importancias_df = pd.DataFrame(
    classificador_arvore.feature_importances_,
    index=features_arvore,
    columns=["importancia"]
).sort_values(
    "importancia",
    ascending=False
)

importancias_df.head(10)


## Avaliando o modelo
Vamos observar as métricas de classificação.


### Regressão logística:

In [ ]:

y_pred = modelo.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
y_train_pred = modelo.predict(X_train)
y_test_pred = modelo.predict(X_test)

print("Treinamento:")
print(accuracy_score(y_train, y_train_pred))

print("\nTeste:")
print(accuracy_score(y_test, y_test_pred))

### Árvore de decisão:

In [ ]:
print(classification_report(
    y_test,
    modelo_arvore.predict(X_test),
    zero_division=0
))

In [ ]:
y_train_pred_arvore = modelo_arvore.predict(X_train)
y_test_pred_arvore = modelo_arvore.predict(X_test)

print("Treinamento:")
print(accuracy_score(y_train, y_train_pred_arvore))

print("\nTeste:")
print(accuracy_score(y_test, y_test_pred_arvore))


### Como interpretar?

- **Precision:** entre as mensagens que o modelo classificou como uma determinada intenção, quantas estavam corretas?
- **Recall:** entre as mensagens que realmente pertenciam a uma intenção, quantas o modelo conseguiu encontrar?
- **F1-score:** combina precision e recall em uma única medida.

### Debate

Um modelo com 100% de acerto neste pequeno dataset significa que temos um chatbot pronto para produção?

**Não.**

O conjunto é pequeno e controlado. Em uma aplicação real, seria necessário avaliar generalização, qualidade das respostas, situações inesperadas, dados novos e outros aspectos.


In [ ]:
acc_lr_treino = accuracy_score(
    y_train,
    modelo.predict(X_train)
)

acc_lr_teste = accuracy_score(
    y_test,
    modelo.predict(X_test)
)

acc_arvore_treino = accuracy_score(
    y_train,
    modelo_arvore.predict(X_train)
)

acc_arvore_teste = accuracy_score(
    y_test,
    modelo_arvore.predict(X_test)
)

print("=== COMPARAÇÃO DOS MODELOS ===")

print("\nRegressão Logística")
print(f"Treinamento: {acc_lr_treino:.2f}")
print(f"Teste:       {acc_lr_teste:.2f}")

print("\nÁrvore de Decisão")
print(f"Treinamento: {acc_arvore_treino:.2f}")
print(f"Teste:       {acc_arvore_teste:.2f}")